In [14]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

In [21]:
results_1l = pd.read_excel("resultados-1l.xlsx")
results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

results = pd.concat(
    [results_1l, results_2l],
    ignore_index=True
)
results = results_2l


In [22]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_theta,R2diff_ZZx1_theta,R2_ZZx2_theta,R2diff_ZZx2_theta,...,R2_LSG_1_theta,R2diff_LSG_1_theta,R2_LSG_2_theta,R2diff_LSG_2_theta,R2_ZZx1_inv_theta,R2diff_ZZx1_inv_theta,R2_zzx2_inv2_theta,R2diff_zzx2_inv2_theta,R2_semiCirc_theta,R2diff_semiCirc_theta
0,model_arch31-9_r0.01_Ld0.5_Lp0.5_seed2668,"[31, 9]",0.5,0.5,0.01,2668,0.708677,0.557043,0.915348,0.390944,...,-0.079639,0.473519,-2.070070,0.267200,-1.165910,0.242803,-10.137105,0.067175,-19.557043,-0.057024
1,model_arch31-9_r0.01_Ld0.5_Lp0.5_seed4377,"[31, 9]",0.5,0.5,0.01,4377,0.717028,0.606932,0.936157,0.422459,...,0.363038,0.544497,-2.304507,0.344763,-2.357629,0.327433,-14.078534,0.132109,-27.309341,-0.152841
2,model_arch31-9_r0.01_Ld0.5_Lp0.5_seed7301,"[31, 9]",0.5,0.5,0.01,7301,0.874927,0.619037,0.936777,0.398861,...,0.777952,0.578658,-1.559806,0.357269,-0.390839,0.358169,-10.197965,0.107615,-33.465077,-0.293180
3,model_arch31-9_r0.01_Ld0.5_Lp0.5_seed6122,"[31, 9]",0.5,0.5,0.01,6122,-0.629055,0.587534,0.951848,0.400240,...,0.307137,0.590250,-2.075567,0.415681,-4.420488,0.368348,-13.130701,0.099576,-22.976457,-0.133859
4,model_arch31-9_r0.01_Ld0.5_Lp0.5_seed2005,"[31, 9]",0.5,0.5,0.01,2005,0.927774,0.623437,0.930987,0.369480,...,0.854968,0.545618,-1.900352,0.313283,-0.842384,0.302590,-12.122435,0.033354,-33.186747,-0.291799
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3750,model_arch94-64_r0.9_Ld0.7_Lp0.3_seed1716,"[94, 64]",0.7,0.3,0.90,1716,0.917239,0.679375,0.801278,0.482787,...,0.802490,0.595171,-1.352319,0.373398,-0.880326,0.450889,-10.007006,0.179014,-37.644907,-0.369949
3751,model_arch94-64_r0.9_Ld0.7_Lp0.3_seed5358,"[94, 64]",0.7,0.3,0.90,5358,0.935653,0.796518,0.808687,0.475581,...,0.777894,0.624707,-3.765155,0.364903,-4.045055,0.463253,-21.247402,-0.077030,-38.954690,-0.382252
3752,model_arch94-64_r0.9_Ld0.7_Lp0.3_seed5928,"[94, 64]",0.7,0.3,0.90,5928,0.904866,0.627094,0.722774,0.473950,...,0.843056,0.557651,-2.869551,0.318823,-0.259116,0.384334,-8.726999,0.134795,-30.275763,-0.242740
3753,model_arch94-64_r0.9_Ld0.7_Lp0.3_seed5969,"[94, 64]",0.7,0.3,0.90,5969,-0.030162,0.660684,0.962224,0.488086,...,0.245412,0.588647,-1.589548,0.396917,-4.562926,0.460647,-10.816230,0.262000,-28.412967,-0.172306


In [23]:
# 🔹 categorização dos sets (baseada nos comentários originais)

SETS_CATEGORY = {
    "ZZx1":  "Train",
    "ZZx2":     "Val",
    "ZZy1":     "Test",
    "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZxReto":  "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33

for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
2754,model_arch94-31_r0.9_Ld0.3_Lp0.7_seed5071,"[94, 31]",0.514552,0.479189,-2.500407,-0.814473
3049,model_arch94-41_r0.01_Ld0.3_Lp0.7_seed5071,"[94, 41]",0.538028,0.543446,-2.638222,-0.822874
3381,model_arch94-52_r0.9_Ld0.3_Lp0.7_seed5358,"[94, 52]",0.608616,0.525570,-2.760733,-0.846819
3350,model_arch94-51_r0.9_Ld0.3_Lp0.7_seed1716,"[94, 51]",0.605924,0.557058,-2.842315,-0.862985
3108,model_arch94-43_r0.01_Ld0.3_Lp0.7_seed5969,"[94, 43]",0.480850,0.536764,-2.701271,-0.879513



📊 MÉTRICAS COMPLETAS - TOP 5 (theta)


,model,Neurons,R2_ZZx1_theta,R2_ZZx2_theta,R2_ZZy1_theta,R2_ZZy2_theta,R2_LSG_1_theta,R2_LSG_2_theta,R2_ZZx1_inv_theta,R2_ZZxReto_theta,R2_semiCirc_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
2754,model_arch94-31_r0.9_Ld0.3_Lp0.7_seed5071,"[94, 31]",0.514552,0.479189,-0.562409,-8.814811,-5.234725,-0.619412,-0.304284,0.131777,-2.098982,0.514552,0.479189,-2.500407,-0.814473
3049,model_arch94-41_r0.01_Ld0.3_Lp0.7_seed5071,"[94, 41]",0.538028,0.543446,-1.403530,-8.891398,-4.382130,-0.613407,-0.317955,0.097619,-2.956753,0.538028,0.543446,-2.638222,-0.822874
3381,model_arch94-52_r0.9_Ld0.3_Lp0.7_seed5358,"[94, 52]",0.608616,0.525570,-2.531099,-8.673570,-2.987812,-0.695921,-0.143774,0.378035,-4.670987,0.608616,0.525570,-2.760733,-0.846819
3350,model_arch94-51_r0.9_Ld0.3_Lp0.7_seed1716,"[94, 51]",0.605924,0.557058,-2.989535,-8.552234,-2.894652,-0.730935,-0.135095,0.256822,-4.850578,0.605924,0.557058,-2.842315,-0.862985
3108,model_arch94-43_r0.01_Ld0.3_Lp0.7_seed5969,"[94, 43]",0.480850,0.536764,-0.849364,-9.097618,-5.440518,-0.550003,-0.556314,-0.120475,-2.294604,0.480850,0.536764,-2.701271,-0.879513


In [24]:
final_table.to_excel("BestModels-2l.xlsx")